In [28]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

data = pd.read_csv("flight_data_ERA5.csv")

In [48]:

import pandas as pd

data = pd.read_csv("flight_data_ERA5.csv")

significant_features = [
    "wind_speed_10m",
    "relative_humidity_2m",
    "cloud_cover",
    "Past_temperature",
    "Past_humidity",
    "Past_pressure",
    "Past_cloudcover",
    "Market Share",
    "Passenger Load Factor",
    "OTP Index",
    "Hourly_Airport_Departures",
    "Distance",
    "Airport_Congestion_Index"
]

# Select the 13 features directly
X = data[significant_features].copy()

# Convert features to numeric
X = X.apply(pd.to_numeric, errors="coerce")

# Binary target
y = (data["Departure Delay"] > 0).astype(int)

print("X shape:", X.shape)
print("Missing values:")
print(X.isna().sum())
print("Class distribution:")
print(y.value_counts(normalize=True) * 100)


X shape: (10642, 13)
Missing values:
wind_speed_10m                0
relative_humidity_2m          0
cloud_cover                   0
Past_temperature              0
Past_humidity                 0
Past_pressure                 0
Past_cloudcover               0
Market Share                  0
Passenger Load Factor         0
OTP Index                     0
Hourly_Airport_Departures    69
Distance                      0
Airport_Congestion_Index     69
dtype: int64
Class distribution:
Departure Delay
0    51.409509
1    48.590491
Name: proportion, dtype: float64


In [50]:
# 5-fold cross-validation
kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# KNN with 5 neighbors
knn = make_pipeline(
    SimpleImputer(strategy="median", keep_empty_features=True),
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=21)
)

# Cross-validated predictions
y_pred = cross_val_predict(knn, X, y, cv=kfold)

# Calculate metrics
tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

print(f"Overall Accuracy: {accuracy_score(y, y_pred):.2%}")
print(f"F1 Score: {f1_score(y, y_pred):.4f}")
print(f"False Positive Rate: {fp / (fp + tn):.2%}")
print(f"False Negative Rate: {fn / (fn + tp):.2%}")
print(f"Recall: {tp / (tp + fn):.2%}")
print("\nConfusion Matrix:")
print(confusion_matrix(y, y_pred))


from sklearn.model_selection import cross_validate
import numpy as np

scores = cross_validate(
    knn,
    X,
    y,
    cv=kfold,
    scoring=["accuracy", "f1"],
    return_train_score=True
)

print("Training Accuracy:",
      f"{scores['train_accuracy'].mean():.2%}")

print("Testing Accuracy:",
      f"{scores['test_accuracy'].mean():.2%}")

print("Training F1:",
      f"{scores['train_f1'].mean():.4f}")

print("Testing F1:",
      f"{scores['test_f1'].mean():.4f}")

#HIGHEST 66.43

Overall Accuracy: 66.43%
F1 Score: 0.6496
False Positive Rate: 31.29%
False Negative Rate: 35.97%
Recall: 64.03%

Confusion Matrix:
[[3759 1712]
 [1860 3311]]
Training Accuracy: 70.59%
Testing Accuracy: 66.43%
Training F1: 0.6933
Testing F1: 0.6495


In [33]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

k_values = [3, 5, 7, 9, 11]

for k in k_values:
    knn = make_pipeline(
        SimpleImputer(strategy="median", keep_empty_features=True),
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=k)
    )

    auc_scores = cross_val_score(
        knn,
        X,
        y,
        cv=kfold,
        scoring="roc_auc"
    )

    accuracy_scores = cross_val_score(
        knn,
        X,
        y,
        cv=kfold,
        scoring="accuracy"
    )

    print(
        f"k={k:2d} | "
        f"AUC={auc_scores.mean():.4f} | "
        f"Accuracy={accuracy_scores.mean():.2%}"
    )

k= 3 | AUC=0.6878 | Accuracy=64.78%
k= 5 | AUC=0.7087 | Accuracy=65.93%
k= 7 | AUC=0.7161 | Accuracy=66.19%
k= 9 | AUC=0.7197 | Accuracy=66.39%
k=11 | AUC=0.7232 | Accuracy=66.69%


In [40]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score

kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf = make_pipeline(
    SimpleImputer(strategy="median", keep_empty_features=True),
    RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )
)

# Cross-validated predictions
y_pred = cross_val_predict(rf, X, y, cv=kfold)

# Cross-validated probabilities
y_prob = cross_val_predict(
    rf, X, y, cv=kfold, method="predict_proba"
)[:, 1]

tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

print(f"Overall Accuracy: {accuracy_score(y, y_pred):.2%}")
print(f"F1 Score: {f1_score(y, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y, y_prob):.4f}")
print(f"False Positive Rate: {fp / (fp + tn):.2%}")
print(f"False Negative Rate: {fn / (fn + tp):.2%}")
print(f"Recall: {tp / (tp + fn):.2%}")
print("\nConfusion Matrix:")
print(confusion_matrix(y, y_pred))

Overall Accuracy: 69.62%
F1 Score: 0.6866
ROC-AUC: 0.7632
False Positive Rate: 29.30%
False Negative Rate: 31.52%
Recall: 68.48%

Confusion Matrix:
[[3868 1603]
 [1630 3541]]


In [128]:
#test other cutoffs


data = pd.read_csv("flight_data_ERA5.csv")


other_cols = [
    "Market Share",
    "Passenger Load Factor",
    "Distance",
    "OTP Index",
    "Airport_Congestion_Index",
    "Hourly_Airport_Departures"
]


# Same 13 features as reduced Logistic Regression
significant_features = significant_features = [
    "Past_temperature",
    "Past_humidity",
    "Past_pressure",
    "Past_cloudcover",
    "Market Share",
    "Passenger Load Factor",
    "OTP Index",
    "Hourly_Airport_Departures",
    "Distance",
    "Airport_Congestion_Index"
] #best model


X = data[significant_features].copy()

# Original delay > 0 classification
y = (data["Departure Delay"] > 0).astype(int)

print("X shape:", X.shape)
print("Class distribution:")
print(y.value_counts(normalize=True) * 100)

X shape: (10642, 10)
Class distribution:
Departure Delay
0    51.409509
1    48.590491
Name: proportion, dtype: float64


In [ ]:
significant_features = [
    "Past_temperature",
    "Past_humidity",
    "Past_pressure",
    "Past_cloudcover",
    "Market Share",
    "Passenger Load Factor",
    "OTP Index",
    "Hourly_Airport_Departures",
    "Distance",
    "Airport_Congestion_Index"
] #best model

[
    "weather__hourly__windspeedKmph",
    "weather__hourly__humidity",
    "weather__hourly__cloudcover",
    "Past_temperature",
    "Past_humidity",
    "Past_pressure",
    "Past_cloudcover",
    "Market Share",
    "Passenger Load Factor",
    "OTP Index",
    "Hourly_Airport_Departures",
    "weather__hourly__weatherDesc__value_Partly cloudy",
    "weather__hourly__weatherDesc__value_Sunny",
    "Distance",
    "Airport_Congestion_Index"
]

['weather__hourly__windspeedKmph',
 'weather__hourly__humidity',
 'weather__hourly__cloudcover',
 'Past_temperature',
 'Past_humidity',
 'Past_pressure',
 'Past_cloudcover',
 'Market Share',
 'Passenger Load Factor',
 'OTP Index',
 'Hourly_Airport_Departures',
 'weather__hourly__weatherDesc__value_Partly cloudy',
 'weather__hourly__weatherDesc__value_Sunny',
 'Distance',
 'Airport_Congestion_Index']

In [130]:
# 5-fold cross-validation
kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# KNN with 5 neighbors
knn = make_pipeline(
    SimpleImputer(strategy="median", keep_empty_features=True),
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=21)
)

# Cross-validated predictions
y_pred = cross_val_predict(knn, X, y, cv=kfold)

# Calculate metrics
tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

print(f"Overall Accuracy: {accuracy_score(y, y_pred):.2%}")
print(f"F1 Score: {f1_score(y, y_pred):.4f}")
print(f"False Positive Rate: {fp / (fp + tn):.2%}")
print(f"False Negative Rate: {fn / (fn + tp):.2%}")
print(f"Recall: {tp / (tp + fn):.2%}")
print("\nConfusion Matrix:")
print(confusion_matrix(y, y_pred))
#HIGHEST 67.23

#Your KNN results suggest that flights delayed by any amount may be easier to distinguish from on-time flights than flights delayed by more than 15 minutes are from all other flights.

Overall Accuracy: 67.23%
F1 Score: 0.6607
False Positive Rate: 31.27%
False Negative Rate: 34.35%
Recall: 65.65%

Confusion Matrix:
[[3760 1711]
 [1776 3395]]
